# RQ2: raw purchase history, Qwen summary and deterministic summary

Start this notebook from the project folder. `inputs.zip` is the verified frozen development snapshot;
`checkpoint-parts/` contains the pinned public model files. The first code cell lists the paths
and parameters. Live progress and final outputs go to `results/rq2_h100_v2/`.

In [ ]:
from collections import Counter
from pathlib import Path
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import random
import subprocess
import sys
import time
import zipfile


project = Path.cwd()
scratch = Path('/tmp/llm4rec-session3-rq2')
data = project / 'data/inputs_v1'
results = project / 'results/rq2_h100_v2'
model_path = scratch / 'models/Qwen3-1.7B-70d244cc'
model_revision = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
protocol_hash = 'd46b45deaf2c2f955c52fa020082971947d5ac8e27073a90d1c5384f648af697'
order_hash = '8047bbccbbe66f0bc420df36b06484c4f0888fe18deb8caf0897f1d4903acb04'
bundle_hash = '7ee8436bb131d3927d71f44f0209bf0b2ddcad3742d06d962bc33eaa2ab7b189'
physical_gpu, batch_size = 3, 8
summary_limit, rank_limit, context_limit = 120, 60, 8192
offline_dependencies = Path('/home/rshangareevil/REC-23097-20261116/llm4rec_s2_rq2/offline_packages_v1')
seed = 0


def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def update_json(path, value):
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)


def progress(stage, **details):
    update_json(results / 'progress.json', {'stage': stage, 'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                                           **details})


def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


preparation_start = time.perf_counter()
results.mkdir(parents=True, exist_ok=False)
if not data.exists():
    with zipfile.ZipFile(scratch / 'inputs.zip') as archive:
        assert digest(scratch / 'inputs.zip') == bundle_hash
        manifest = json.loads(archive.read('input-manifest.json'))
        assert set(archive.namelist()) == set(manifest) | {'input-manifest.json'}
        assert all((data / name).resolve().is_relative_to(data.resolve()) for name in archive.namelist())
        archive.extractall(data)
manifest = json.loads((data / 'input-manifest.json').read_text())
assert all(digest(data / name) == sha for name, sha in manifest.items())
protocol = json.loads((data / 'starter200/protocol.json').read_text())
contexts = read_jsonl(data / 'starter200/contexts.jsonl')
assert protocol['protocol_hash'] == protocol_hash
assert hashlib.sha256(json.dumps({key: value for key, value in protocol.items() if key != 'protocol_hash'},
                                 sort_keys=True).encode()).hexdigest() == protocol_hash
assert [ctx['user_id'] for ctx in contexts] == protocol['users'] and len(contexts) == 200
candidate_lists = [[item['id'] for item in ctx['candidate_assets']] for ctx in contexts]
assert all(len(candidates) == len(set(candidates)) == 50 for candidates in candidate_lists)
assert hashlib.sha256('\n'.join(','.join(row) for row in candidate_lists).encode()).hexdigest() == order_hash
assert all(ctx['cutoff'] == '2021-06-01' and
           all(asset['metadata_timestamp'][:10] < ctx['cutoff'] for asset in ctx['history'] + ctx['candidate_assets'])
           for ctx in contexts)
shared = load_module('shared_rerank', data / 'shared_rerank.py')
data_seconds = time.perf_counter() - preparation_start
print(json.dumps({'users': len(contexts), 'candidate_order_sha256': order_hash, 'data_seconds': data_seconds}))

In [ ]:
part_root = scratch / 'checkpoint-parts'
model_start = time.perf_counter()
canonical = {item['path']: item for item in json.loads((data / 'hf-tree.json').read_text()) if item['type'] == 'file'}
assert json.loads((data / 'hf-api.json').read_text())['sha'] == model_revision
required_model_files = {name for name in canonical if name.endswith(('.json', '.txt', '.safetensors'))}


def verify_model_blob(name, path):
    item = canonical[name]
    assert path.stat().st_size == item['size'], name
    if 'lfs' in item:
        assert digest(path) == item['lfs']['oid'], name
    else:
        assert hashlib.sha1(f'blob {item["size"]}\0'.encode() + path.read_bytes()).hexdigest() == item['oid'], name


if not model_path.exists():
    deadline = time.monotonic() + 25 * 60
    while True:
        if (part_root / 'manifest.json').exists():
            transfer = json.loads((part_root / 'manifest.json').read_text())
            assert transfer['revision'] == model_revision
            pieces = {row['name']: row['parts'] for row in transfer['files'] if row['name'] in required_model_files}
            assert set(pieces) == required_model_files
            waiting = [Path(part['path']).name for parts in pieces.values() for part in parts
                       if not (part_root / Path(part['path']).name).is_file()]
            if not waiting:
                break
            progress('waiting for checkpoint upload', pieces_missing=len(waiting))
        else:
            progress('waiting for checkpoint manifest')
        if time.monotonic() > deadline:
            raise TimeoutError('The verified model transfer did not finish within 25 minutes')
        time.sleep(8)
    pending_model = model_path.with_name(model_path.name + '.assembling')
    pending_model.mkdir(parents=True, exist_ok=False)
    progress('verifying checkpoint')
    for name, parts in sorted(pieces.items()):
        with (pending_model / name).open('xb') as destination:
            for entry in parts:
                piece = part_root / Path(entry['path']).name
                assert piece.stat().st_size == entry['bytes']
                with piece.open('rb') as source:
                    while block := source.read(4 * 1024**2):
                        destination.write(block)
        verify_model_blob(name, pending_model / name)
    pending_model.rename(model_path)
else:
    for name in required_model_files:
        verify_model_blob(name, model_path / name)
checkpoint_seconds = time.perf_counter() - model_start
progress('checkpoint verified; checking GPU 3')

gpu_row = subprocess.check_output(['nvidia-smi', '-i', str(physical_gpu),
    '--query-gpu=index,uuid,name,memory.used,utilization.gpu', '--format=csv,noheader,nounits'], text=True).strip()
index, device_uuid, device_name, used, utilization = [value.strip() for value in gpu_row.split(',')]
assert int(index) == physical_gpu and 'H100' in device_name
if int(used) >= 256 or int(utilization) != 0:
    raise RuntimeError(f'GPU 3 became busy ({used} MiB, {utilization}%); no other GPU will be used')
os.environ['CUDA_VISIBLE_DEVICES'] = device_uuid
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert (offline_dependencies / 'transformers').is_dir()
sys.path.insert(0, str(offline_dependencies))
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.device_count() == 1
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
random.seed(seed)
torch.manual_seed(seed)
device = torch.device('cuda:0')
start_load = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, trust_remote_code=False)
tokenizer.padding_side = 'left'
assert tokenizer.pad_token_id is not None
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True, trust_remote_code=False,
                                             torch_dtype=torch.float32, attn_implementation='sdpa').to(device).eval()
torch.cuda.synchronize()
eos_ids = model.generation_config.eos_token_id
if isinstance(eos_ids, int):
    eos_ids = [eos_ids]
assert eos_ids and model.device == device
model_metadata = {'model': 'Qwen/Qwen3-1.7B', 'revision': model_revision, 'dtype': str(model.dtype),
                  'tf32': False, 'do_sample': False, 'thinking': False, 'seed': seed, 'batch_size': batch_size,
                  'attention': 'sdpa', 'gpu': device_name, 'physical_gpu': physical_gpu, 'visible_gpus': 1,
                  'torch': torch.__version__, 'torch_cuda': torch.version.cuda,
                  'transformers': transformers.__version__, 'python': platform.python_version(),
                  'os': platform.platform(), 'load_seconds': time.perf_counter() - start_load,
                  'data_preparation_seconds': data_seconds, 'checkpoint_wait_and_verification_seconds': checkpoint_seconds,
                  'protocol_hash': protocol_hash, 'candidate_order_sha256': order_hash}
update_json(results / 'model.json', model_metadata)
progress('model loaded on GPU 3')

In [ ]:
def generate_stage(stage, message_lists, token_limit):
    output = results / f'{stage}.jsonl'
    stage_start = time.perf_counter()
    rows = []
    with output.open('x', encoding='utf-8') as destination:
        for begin in range(0, len(message_lists), batch_size):
            batch = message_lists[begin:begin + batch_size]
            rendered = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                       enable_thinking=False) for messages in batch]
            tokenized = tokenizer(rendered, add_special_tokens=False)['input_ids']
            accepted = [offset for offset, ids in enumerate(tokenized) if len(ids) + token_limit <= context_limit]
            returned = {}
            generation_seconds = 0.0
            if accepted:
                encoded = tokenizer.pad({'input_ids': [tokenized[offset] for offset in accepted]},
                                        padding=True, return_attention_mask=True, return_tensors='pt').to(device)
                input_width = encoded['input_ids'].shape[-1]
                torch.cuda.synchronize()
                generation_start = time.perf_counter()
                with torch.inference_mode():
                    generated = model.generate(**encoded, max_new_tokens=token_limit, do_sample=False,
                                               pad_token_id=tokenizer.pad_token_id)
                torch.cuda.synchronize()
                generation_seconds = time.perf_counter() - generation_start
                for offset, output_tokens in zip(accepted, generated[:, input_width:].tolist()):
                    stop = next((i for i, tok in enumerate(output_tokens) if tok in eos_ids), None)
                    content = output_tokens if stop is None else output_tokens[:stop]
                    returned[offset] = (tokenizer.decode(content, skip_special_tokens=True).strip(),
                                        len(output_tokens) if stop is None else stop + 1, stop is None)
            for offset, messages in enumerate(batch):
                response, completion_tokens, capped = returned.get(offset, (None, 0, False))
                record = {'position': begin + offset + 1, 'messages': messages, 'text': response,
                          'input_tokens': len(tokenized[offset]), 'output_tokens': completion_tokens,
                          'hit_token_cap': capped,
                          'error': None if offset in returned else 'context_limit_exceeded'}
                rows.append(record)
                destination.write(json.dumps(record, ensure_ascii=False) + '\n')
            destination.flush()
            progress(stage, completed=len(rows), total=len(message_lists),
                     stage_seconds=round(time.perf_counter() - stage_start, 3),
                     last_batch_generation_seconds=round(generation_seconds, 3))
    metrics = {'requests': len(rows), 'max_new_tokens': token_limit, 'wall_seconds': time.perf_counter() - stage_start,
               'input_tokens': sum(row['input_tokens'] for row in rows),
               'output_tokens': sum(row['output_tokens'] for row in rows),
               'token_cap_count': sum(row['hit_token_cap'] for row in rows),
               'context_limit_errors': sum(bool(row['error']) for row in rows)}
    update_json(results / f'{stage}-runtime.json', metrics)
    print(stage, len(rows), round(metrics['wall_seconds'], 2), flush=True)
    return rows


def shared_messages(prompt):
    system, user = prompt
    return [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]


def compress(ctx):
    history = ctx['history'][:shared.HISTORY_CAP]
    prefix = f"Among the {len(history)} shown assets, the investor made {sum(a['buy_count'] for a in history)} Buy transactions."
    groups = []
    for field, label in (('assetCategory', 'Category by Buy count'), ('marketID', 'Market by Buy count')):
        counts = Counter()
        for asset in history:
            attribute = next((value.split('=', 1)[1] for value in asset['attributes'] if value.startswith(field + '=')), 'unknown')
            counts[attribute] += asset['buy_count']
        groups.append((label, [f'{key}: {value}' for key, value in sorted(counts.items(), key=lambda pair: (-pair[1], pair[0]))]))
    groups.append(('Top ISINs by Buy count', [f"{a['id']}: {a['buy_count']}" for a in history]))
    accepted = [[] for _ in groups]

    def render():
        return ' '.join([prefix] + [f'{label}: {", ".join(values)}.' for (label, _), values in zip(groups, accepted) if values])

    for rank in range(max(len(full) for _, full in groups)):
        for group, (_, full) in enumerate(groups):
            if rank >= len(full):
                continue
            accepted[group].append(full[rank])
            if len(tokenizer.encode(render(), add_special_tokens=False)) > summary_limit:
                accepted[group].pop()
    summary = render()
    assert len(tokenizer.encode(summary, add_special_tokens=False)) <= summary_limit
    return summary


def shared_ids(text, candidates):
    if text is None:
        return candidates[:10], False, True
    try:
        numbers = shared.parse_ranking(text, len(candidates))
    except TypeError:
        numbers = []
    if len(numbers) != 10:
        return candidates[:10], False, True
    try:
        original = json.loads(text)
    except json.JSONDecodeError:
        original = None
    exact = isinstance(original, dict) and original.get('ranking') == numbers and set(original) == {'ranking'}
    return [candidates[number - 1] for number in numbers], not exact, False


summary_requests = [shared_messages(shared.summary_prompt(ctx, 'C', {})) for ctx in contexts]
summary_rows = generate_stage('llm-summary', summary_requests, summary_limit)
compression_start = time.perf_counter()
template_summaries = [compress(ctx) for ctx in contexts]
update_json(results / 'template-generation.json', {'summaries': template_summaries,
            'wall_seconds': time.perf_counter() - compression_start})
information = {'raw': [shared.history_block(ctx) for ctx in contexts],
               'summary': [row['text'] or '[No summary returned.]' for row in summary_rows],
               'template': template_summaries}
for variant, blocks in information.items():
    requests = [shared_messages(shared.rank_prompt(ctx, info)) for ctx, info in zip(contexts, blocks)]
    generations = generate_stage(variant, requests, rank_limit)
    ranks = []
    for row, candidates in zip(generations, candidate_lists):
        ranking, repair, fallback = shared_ids(row['text'], candidates)
        ranks.append({'position': row['position'], 'ranking': ranking, 'repair': repair, 'fallback': fallback})
    update_json(results / f'{variant}-rankings.json', ranks)
progress('matched variants finished')

In [ ]:
far = data / 'far-trans-data'
if not far.exists():
    with zipfile.ZipFile(data / 'FAR-Trans.zip') as archive:
        for name in protocol['source_sha256']:
            archive.extract('FAR-Trans/' + name, far)
far = far / 'FAR-Trans'
assert all(digest(far / name) == sha for name, sha in protocol['source_sha256'].items())
starter = load_module('course_starter', data / 'course/far-trans/starter.py')
sys.path.insert(0, str(data / 'course/far-trans-llm'))
provided = load_module('provided_ranker', data / 'course/far-trans-llm/ranker.py')
assets, _, history, _, _, _ = starter.load_data(far, protocol['cutoff'], 30)
events = starter.scan_recent_events(far, protocol['cutoff'], assets)
provided_requests = [[{'role': 'user', 'content': provided.build_prompt(ctx, assets, history[ctx['user_id']],
                            events.get(ctx['user_id'], ()), history_events=15, shuffle_seed=None)}] for ctx in contexts]
provided_rows = generate_stage('provided', provided_requests, 512)
provided_ranks = []
for row, candidates in zip(provided_rows, candidate_lists):
    repair, fallback = False, False
    try:
        ranking, repair = provided.parse_ranking(row['text'], set(candidates))
        starter.validate_ranking(ranking, candidates)
    except (ValueError, TypeError, KeyError, RuntimeError, AttributeError):
        ranking, fallback = candidates[:10], True
    provided_ranks.append({'position': row['position'], 'ranking': ranking, 'repair': repair, 'fallback': fallback})
update_json(results / 'provided-rankings.json', provided_ranks)
progress('inference finished', users=len(contexts))